In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType,DateType
)

spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [2]:
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("mebership_level", StringType(), True)
])

In [4]:
customer_df = spark.read \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(customers_schema) \
    .csv("s3a://ment4/ment5/customers.csv")

In [5]:
customer_df.show()

+-----------+----------------+-----------+-----------------+---------------+
|customer_id|       full_name|       city|registration_date|mebership_level|
+-----------+----------------+-----------+-----------------+---------------+
|       C001| Sevinc Huseynov|Mingachevir|       2024-08-26|         Bronze|
|       C002|   Nigar Karimov|       Baku|       2023-01-19|         Bronze|
|       C003|    Ramin Rzayev|       Baku|       2021-06-26|       Platinum|
|       C004|  Aytac Mammadov|     baku  |       2021-07-05|       Platinum|
|       C005|    Rauf Bagirov|       Baku|       2022-04-03|         Bronze|
|       C006|   Javid Bagirov|Mingachevir|       2021-04-12|         Silver|
|       C007|    Rauf Karimov|      Sheki|       2021-09-30|           Gold|
|       C008|  Aytac Huseynov|   Lankaran|       2021-08-30|           Gold|
|       C009| Ulviyya Safarov|       BAKU|       2021-07-31|         Silver|
|       C010| Orkhan Mammadov|   Lankaran|       2021-05-09|         Bronze|

In [8]:
customer_df.createOrReplaceTempView("customers")

In [9]:
customers_clean = spark.sql("""
    SELECT
        customer_id,
        full_name,
        INITCAP(TRIM(city)) AS city,
        registration_date,
        mebership_level
    FROM customers
""")

customers_clean.show()

+-----------+----------------+-----------+-----------------+---------------+
|customer_id|       full_name|       city|registration_date|mebership_level|
+-----------+----------------+-----------+-----------------+---------------+
|       C001| Sevinc Huseynov|Mingachevir|       2024-08-26|         Bronze|
|       C002|   Nigar Karimov|       Baku|       2023-01-19|         Bronze|
|       C003|    Ramin Rzayev|       Baku|       2021-06-26|       Platinum|
|       C004|  Aytac Mammadov|       Baku|       2021-07-05|       Platinum|
|       C005|    Rauf Bagirov|       Baku|       2022-04-03|         Bronze|
|       C006|   Javid Bagirov|Mingachevir|       2021-04-12|         Silver|
|       C007|    Rauf Karimov|      Sheki|       2021-09-30|           Gold|
|       C008|  Aytac Huseynov|   Lankaran|       2021-08-30|           Gold|
|       C009| Ulviyya Safarov|       Baku|       2021-07-31|         Silver|
|       C010| Orkhan Mammadov|   Lankaran|       2021-05-09|         Bronze|

In [11]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity",  DoubleType(), True),
    StructField("order_date", DateType(), True),
    StructField("status", StringType(), True)
])

In [12]:
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_price",  DoubleType(), True),
    StructField("supplier", StringType(), True)    
])

In [14]:
order_df = spark.read \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(orders_schema) \
    .csv("s3a://ment4/ment5/orders.csv")

In [32]:
order_df.show()

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|   O0001|       C003|      P003|     8.0|2024-07-24|Delivered|
|   O0002|       C005|      P014|     7.0|2024-10-08|Delivered|
|   O0003|       C014|      P006|     7.0|2024-04-28|Delivered|
|   O0004|       C006|      P003|     4.0|2024-04-29|Delivered|
|   O0005|       C019|      P003|     5.0|2024-05-24|Delivered|
|   O0006|       C014|      P009|     6.0|2024-11-08|  Shipped|
|   O0007|       C005|      P012|     1.0|2024-08-21|Cancelled|
|   O0008|       C025|      P014|     7.0|2024-07-22|Delivered|
|   O0009|       C004|      P008|     7.0|2024-02-01|Delivered|
|   O0010|       C007|      P008|     3.0|2024-02-26|Delivered|
|   O0011|       C002|      P002|     1.0|2024-10-17|Delivered|
|   O0012|       C004|      P006|     1.0|2024-02-06|Cancelled|
|   O0013|       C020|      P007|     3.

In [15]:
product_df = spark.read \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(products_schema) \
    .csv("s3a://ment4/ment5/products.csv")

In [33]:
product_df.show()

+----------+------------+-----------+----------+-----------+
|product_id|product_name|   category|unit_price|   supplier|
+----------+------------+-----------+----------+-----------+
|      P001|      Laptop|Electronics|    1450.0| TechSupply|
|      P002|  Smartphone|Electronics|     899.5| TechSupply|
|      P003|     Monitor|Electronics|     320.0|ScreenWorks|
|      P004|    Keyboard|Electronics|      45.9|ScreenWorks|
|      P005|  Headphones|Electronics|    129.99|  AudioPlus|
|      P006|Office Chair|  furniture|     210.0|  HomeStyle|
|      P007|        Desk|  Furniture|     340.0|  HomeStyle|
|      P008|   Bookshelf|  Furniture|     175.5|  WoodCraft|
|      P009|        Lamp|  Furniture|      65.0|  WoodCraft|
|      P010|    Notebook| Stationery|       4.5|    PaperCo|
|      P011|     Pen Set| Stationery|      NULL|    PaperCo|
|      P012|    Backpack|Accessories|      89.0|  BagMakers|
|      P013|Water Bottle|Accessories|      22.5|  BagMakers|
|      P014|   Mouse Pad

In [34]:
order_df.createOrReplaceTempView("orders")
product_df.createOrReplaceTempView("products")

In [17]:
customers_dup = spark.sql("""
    SELECT *, COUNT(*) AS cnt
    FROM customers
    GROUP BY ALL
    HAVING COUNT(*) > 1
""")

customers_dup.show()

+-----------+------------+-----------+-----------------+---------------+---+
|customer_id|   full_name|       city|registration_date|mebership_level|cnt|
+-----------+------------+-----------+-----------------+---------------+---+
|       C003|Ramin Rzayev|       Baku|       2021-06-26|       Platinum|  2|
|       C011|Melek Rzayev|Mingachevir|       2024-10-25|       Platinum|  2|
+-----------+------------+-----------+-----------------+---------------+---+



In [25]:
customers_clean = customer_df.dropDuplicates()

In [35]:
orders_du = spark.sql("""
    SELECT *, COUNT(*) AS cnt
    FROM orders
    GROUP BY ALL
    HAVING COUNT(*) > 1
""")

orders_du.show()

+--------+-----------+----------+--------+----------+-------+---+
|order_id|customer_id|product_id|quantity|order_date| status|cnt|
+--------+-----------+----------+--------+----------+-------+---+
|   O0006|       C014|      P009|     6.0|2024-11-08|Shipped|  2|
+--------+-----------+----------+--------+----------+-------+---+



In [36]:
products_du = spark.sql("""
    SELECT *, COUNT(*) AS cnt
    FROM products
    GROUP BY ALL
    HAVING COUNT(*) > 1
""")

products_du.show()

+----------+------------+--------+----------+--------+---+
|product_id|product_name|category|unit_price|supplier|cnt|
+----------+------------+--------+----------+--------+---+
+----------+------------+--------+----------+--------+---+



In [39]:
orders_clean = order_df.dropDuplicates()

In [40]:
orders_clean.show()

+--------+-----------+----------+--------+----------+----------+
|order_id|customer_id|product_id|quantity|order_date|    status|
+--------+-----------+----------+--------+----------+----------+
|   O0055|       C014|      P002|     4.0|2024-06-04|Processing|
|   O0046|       C022|      P009|     8.0|2024-02-24| Cancelled|
|   O0066|       C017|      P010|     8.0|2024-06-16| Delivered|
|   O0021|       C006|      P006|     4.0|2024-09-29| Delivered|
|   O0035|       C001|      P003|     8.0|2024-03-15|   Shipped|
|   O0048|       C004|      P009|     8.0|2024-10-14| Delivered|
|   O0065|       C014|      P014|     5.0|2024-07-26| Delivered|
|   O0034|       C003|      P012|     3.0|2024-03-28|  Returned|
|   O0040|       C018|      P007|     3.0|2024-02-01| Cancelled|
|   O0049|       C003|      P008|     6.0|2024-11-09|  Returned|
|   O0064|       C002|      P015|     3.0|2024-05-18|Processing|
|   O0001|       C003|      P003|     8.0|2024-07-24| Delivered|
|   O0060|       C011|   

In [38]:
product_clean = product_df.dropDuplicates()

In [41]:
product_clean.show()

+----------+------------+-----------+----------+-----------+
|product_id|product_name|   category|unit_price|   supplier|
+----------+------------+-----------+----------+-----------+
|      P006|Office Chair|  furniture|     210.0|  HomeStyle|
|      P007|        Desk|  Furniture|     340.0|  HomeStyle|
|      P015|      Webcam|Electronics|     110.0|  AudioPlus|
|      P005|  Headphones|Electronics|    129.99|  AudioPlus|
|      P004|    Keyboard|Electronics|      45.9|ScreenWorks|
|      P008|   Bookshelf|  Furniture|     175.5|  WoodCraft|
|      P001|      Laptop|Electronics|    1450.0| TechSupply|
|      P003|     Monitor|Electronics|     320.0|ScreenWorks|
|      P013|Water Bottle|Accessories|      22.5|  BagMakers|
|      P002|  Smartphone|Electronics|     899.5| TechSupply|
|      P009|        Lamp|  Furniture|      65.0|  WoodCraft|
|      P012|    Backpack|Accessories|      89.0|  BagMakers|
|      P010|    Notebook| Stationery|       4.5|    PaperCo|
|      P014|   Mouse Pad

In [53]:
product_clean.createOrReplaceTempView("product_cl")

In [45]:
orders_clean.createOrReplaceTempView("order_cl")

In [47]:
orders_minus = spark.sql("""
    SELECT *
    FROM order_cl
    WHERE quantity < 0
       OR quantity IS NULL
""")

orders_minus.show()

+--------+-----------+----------+--------+----------+----------+
|order_id|customer_id|product_id|quantity|order_date|    status|
+--------+-----------+----------+--------+----------+----------+
|   O0038|       C005|      P007|    -2.0|2024-04-18| Delivered|
|   O0014|       C012|      P010|    -2.0|2024-08-30| Delivered|
|   O0067|       C002|      P013|    NULL|2024-08-05| Cancelled|
|   O0022|       C017|      P006|    NULL|2024-11-09|Processing|
|   O0051|       C018|      P013|    NULL|2024-09-16|  Returned|
+--------+-----------+----------+--------+----------+----------+



In [50]:
orders_clean = spark.sql("""
    SELECT
        order_id,
        customer_id,
        product_id,
        order_date,
        COALESCE(quantity, 0) AS quantity,
        status
    FROM order_cl
    WHERE quantity >= 0 OR quantity IS NULL
""")

orders_clean.show()

+--------+-----------+----------+----------+--------+----------+
|order_id|customer_id|product_id|order_date|quantity|    status|
+--------+-----------+----------+----------+--------+----------+
|   O0055|       C014|      P002|2024-06-04|     4.0|Processing|
|   O0046|       C022|      P009|2024-02-24|     8.0| Cancelled|
|   O0066|       C017|      P010|2024-06-16|     8.0| Delivered|
|   O0021|       C006|      P006|2024-09-29|     4.0| Delivered|
|   O0035|       C001|      P003|2024-03-15|     8.0|   Shipped|
|   O0048|       C004|      P009|2024-10-14|     8.0| Delivered|
|   O0065|       C014|      P014|2024-07-26|     5.0| Delivered|
|   O0034|       C003|      P012|2024-03-28|     3.0|  Returned|
|   O0040|       C018|      P007|2024-02-01|     3.0| Cancelled|
|   O0049|       C003|      P008|2024-11-09|     6.0|  Returned|
|   O0064|       C002|      P015|2024-05-18|     3.0|Processing|
|   O0001|       C003|      P003|2024-07-24|     8.0| Delivered|
|   O0060|       C011|   

In [58]:
products_clean = spark.sql("""
    SELECT
        product_id,
        product_name,
        category,
        COALESCE(
            unit_price,
            AVG(unit_price) OVER (PARTITION BY category)
        ) AS unit_price
    FROM product_cl
""")

products_clean.show()

+----------+------------+-----------+----------+
|product_id|product_name|   category|unit_price|
+----------+------------+-----------+----------+
|      P013|Water Bottle|Accessories|      22.5|
|      P012|    Backpack|Accessories|      89.0|
|      P014|   Mouse Pad|Accessories|      15.0|
|      P015|      Webcam|Electronics|     110.0|
|      P005|  Headphones|Electronics|    129.99|
|      P004|    Keyboard|Electronics|      45.9|
|      P001|      Laptop|Electronics|    1450.0|
|      P003|     Monitor|Electronics|     320.0|
|      P002|  Smartphone|Electronics|     899.5|
|      P007|        Desk|  Furniture|     340.0|
|      P008|   Bookshelf|  Furniture|     175.5|
|      P009|        Lamp|  Furniture|      65.0|
|      P010|    Notebook| Stationery|       4.5|
|      P011|     Pen Set| Stationery|       4.5|
|      P006|Office Chair|  furniture|     210.0|
+----------+------------+-----------+----------+



In [52]:
missing_customers = spark.sql("""
    SELECT o.*
    FROM orders o
    LEFT ANTI JOIN customers c
        ON o.customer_id = c.customer_id
""")

missing_customers.show()

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|   O0030|       C999|      P013|     2.0|2024-03-02|Cancelled|
+--------+-----------+----------+--------+----------+---------+

